# External Framework Validation — Anonymeter

**Clean, linear pipeline.** Run top to bottom (Kernel -> Restart & Run All).

This notebook quantifies the privacy risk of the CTGAN and CTAB-GAN+ synthetic
datasets using **Anonymeter** (Giomi et al., 2023), an attack-based framework
that measures the three risks identified by the Article 29 Working Party as key
indicators of factual anonymisation: **singling out**, **linkability**, and
**inference**.

Unlike distance-based metrics (the manual DCR, NNDR analysis) or the distributional
metrics of SynthEval and Synthcity, Anonymeter simulates an adversary. For each
risk it runs three attacks: a *main* attack (the attacker uses the synthetic
data to guess information about the real training records), a *control* attack
(the same against held-out real records), and a *baseline* attack (a naive
attacker guessing at random). Comparing the main attack against the control
separates genuine privacy leakage from what is merely learnable about the
population, and the baseline acts as a sanity check.

To stay consistent with the SynthEval and Synthcity notebooks, the same files
are used: `real_minority_train.csv` as the original (`ori`) real data behind the
generation, `synthetic_ctgan_final.csv` / `synthetic_ctab_final.csv` as the
synthetic data (`syn`), and a minority-only holdout reconstructed from the
canonical 70/30 split as the control (`control`). The control is restricted to
the minority class so that it shares the distribution of `ori`, since the
generators were trained on minority records only.

**Requirements:** environment `tese_anonymeter` (Python 3.10) with
`anonymeter==1.1.0`.


## 0. Configuration and paths

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

from anonymeter.evaluators import (
    SinglingOutEvaluator,
    LinkabilityEvaluator,
    InferenceEvaluator,
)

RANDOM_STATE = 123

# >>> Adjust this path if necessary <<<
DATA_DIR = Path("data")

assert DATA_DIR.exists(), f"Folder not found: {DATA_DIR}"
print("Data folder:", DATA_DIR)


Data folder: data


## 1. Canonical column definitions

Identical to the SynthEval and Synthcity notebooks. Anonymeter operates on the
full record including the target, so the working schema here is the 23 features
plus `def_pay`.


In [2]:
TARGET   = "def_pay"

FEATURES = ["LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE",
            "PAY_1", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6",
            "BILL_AMT1", "BILL_AMT2", "BILL_AMT3", "BILL_AMT4", "BILL_AMT5", "BILL_AMT6",
            "PAY_AMT1", "PAY_AMT2", "PAY_AMT3", "PAY_AMT4", "PAY_AMT5", "PAY_AMT6"]

ALL_COLS = FEATURES + [TARGET]

MONETARY = ["LIMIT_BAL"] + [f"BILL_AMT{i}" for i in range(1, 7)] \
                         + [f"PAY_AMT{i}"  for i in range(1, 7)]

assert len(FEATURES) == 23
print("Working schema:", len(ALL_COLS), "columns (23 features + target)")


Working schema: 24 columns (23 features + target)


## 2. Helper: align a dataframe to the canonical schema

Keeps the 23 features plus the target, enforces column order, and casts the
monetary columns to `float64` so that the integer output of CTAB-GAN+ matches
the other datasets. Anonymeter compares records across `ori`, `syn` and
`control`, so all three must share exactly the same columns in the same order.


In [3]:
def align(df, name):
    d = df.copy()
    assert TARGET in d.columns, f"{name}: '{TARGET}' column is required"
    d = d[ALL_COLS]                       # enforce order; errors if any missing
    d[MONETARY] = d[MONETARY].astype("float64")
    d[TARGET] = d[TARGET].astype(int)
    assert list(d.columns) == ALL_COLS
    print(f"{name:<16} -> {d.shape}")
    return d


## 3. Load the real (ori), synthetic (syn) datasets

`ori` is the real minority partition that the generators were trained on. The
two synthetic sets are the same `*_final.csv` files used in the other
frameworks.


In [4]:
ori       = align(pd.read_csv(DATA_DIR / "real_minority_train.csv"),  "ori (real min)")
syn_ctgan = align(pd.read_csv(DATA_DIR / "synthetic_ctgan_final.csv"), "syn ctgan")
syn_ctab  = align(pd.read_csv(DATA_DIR / "synthetic_ctab_final.csv"),  "syn ctabgan+")

# columns must be perfectly aligned across all three
for d in (syn_ctgan, syn_ctab):
    assert list(d.columns) == list(ori.columns)
print("\nColumns aligned across ori and syn.")


ori (real min)   -> (4645, 24)
syn ctgan        -> (11710, 24)
syn ctabgan+     -> (11710, 24)

Columns aligned across ori and syn.


## 4. Reconstruct the canonical split → minority control

Rebuilds the 70/30 stratified split with `RANDOM_STATE = 123`, verifies it
matches `real_train_full.csv` bit for bit (as in the other notebooks), then
restricts the held-out test partition to the minority class. This minority-only
control shares the distribution of `ori`, which is required for the main-vs-
control comparison to reflect genuine leakage rather than a class-distribution
shift.


In [5]:
df_raw = pd.read_csv(DATA_DIR / "UCI_Credit_Card.csv")

df_c = (df_raw
        .drop(columns=["ID"])
        .rename(columns={"PAY_0": "PAY_1",
                         "default.payment.next.month": TARGET}))
df_c["EDUCATION"] = df_c["EDUCATION"].replace({0: 4, 5: 4, 6: 4})
df_c["MARRIAGE"]  = df_c["MARRIAGE"].replace({0: 3})

X = df_c.drop(columns=TARGET)
y = df_c[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE)

# verify reproduction against the stored training partition
real_full = pd.read_csv(DATA_DIR / "real_train_full.csv")
train_recon = X_train.copy()
train_recon[TARGET] = y_train
train_recon = train_recon[real_full.columns].reset_index(drop=True)
assert train_recon.equals(real_full.reset_index(drop=True)), \
    "Reconstructed split != real_train_full.csv"
print("Split reproduced bit for bit. OK.")

# held-out test set, minority only, aligned to the canonical schema
holdout = X_test.copy()
holdout[TARGET] = y_test
control = align(holdout[holdout[TARGET] == 1], "control (min)")
print("Control (minority holdout):", control.shape)


Split reproduced bit for bit. OK.
control (min)    -> (1991, 24)
Control (minority holdout): (1991, 24)


## 5. Attack configuration

`N_ATTACKS` sets how many target records are attacked. A higher number reduces
the statistical uncertainty on the risk estimate at the cost of runtime. It is
capped by the size of the control set (1991 minority records), so a value below
that is used.

For **linkability**, the auxiliary columns are split into two disjoint groups,
simulating an attacker who holds two separate datasets about the same people and
tries to link them. A natural split for this data is demographic attributes
versus financial behaviour.

For **inference**, each column in turn is treated as the secret, with all
remaining columns as auxiliary knowledge, giving a per-attribute risk profile.

Note: the attacks draw their targets at random and `RANDOM_STATE` is not
passed to the evaluators, so repeated runs give slightly different risks,
within the reported confidence intervals.

In [6]:
N_ATTACKS = 1500          # < len(control) = 1991

# Linkability: two disjoint auxiliary sets (demographic vs financial)
AUX_A = ["SEX", "EDUCATION", "MARRIAGE", "AGE", "LIMIT_BAL"]
AUX_B = [c for c in ALL_COLS if c not in AUX_A]

assert set(AUX_A).isdisjoint(AUX_B)
assert set(AUX_A) | set(AUX_B) == set(ALL_COLS)
print("Linkability aux groups:", len(AUX_A), "vs", len(AUX_B), "columns")
print("N_ATTACKS =", N_ATTACKS)


Linkability aux groups: 5 vs 19 columns
N_ATTACKS = 1500


## 6. Singling Out risk

Measures how often the attacker can craft a predicate that isolates a single
real record ("there is only one person with attributes X, Y and Z"). The
univariate mode uses one attribute at a time; the multivariate mode combines
several. Both are reported. The `risk()` method returns the estimated risk with
a confidence interval.


In [7]:
def singling_out(syn, name, mode, n_cols=None):
    kwargs = dict(ori=ori, syn=syn, control=control, n_attacks=N_ATTACKS)
    if mode == "multivariate":
        kwargs["n_cols"] = n_cols
    ev = SinglingOutEvaluator(**kwargs)
    ev.evaluate(mode=mode)
    r = ev.risk()
    print(f"{name:<12} [{mode:>12}]  risk = {r.value:.4f}  CI = "
          f"({r.ci[0]:.4f}, {r.ci[1]:.4f})")
    return {"model": name, "attack": f"singling_out_{mode}",
            "risk": r.value, "ci_low": r.ci[0], "ci_high": r.ci[1]}

so_rows = []
for name, syn in [("CTGAN", syn_ctgan), ("CTAB-GAN+", syn_ctab)]:
    so_rows.append(singling_out(syn, name, "univariate"))
    so_rows.append(singling_out(syn, name, "multivariate", n_cols=3))


CTGAN        [  univariate]  risk = 0.0000  CI = (0.0000, 0.0018)
CTGAN        [multivariate]  risk = 0.0487  CI = (0.0290, 0.0684)
CTAB-GAN+    [  univariate]  risk = 0.0262  CI = (0.0158, 0.0366)
CTAB-GAN+    [multivariate]  risk = 0.0792  CI = (0.0583, 0.1000)


## 7. Linkability risk

Measures whether an attacker holding two disjoint sets of attributes about the
same individuals can use the synthetic data to link them back together. `n_neighbors`
controls how many nearest synthetic records are considered a successful link.


In [8]:
def linkability(syn, name):
    ev = LinkabilityEvaluator(ori=ori, syn=syn, control=control,
                              n_attacks=N_ATTACKS,
                              aux_cols=(AUX_A, AUX_B),
                              n_neighbors=10)
    ev.evaluate()
    r = ev.risk()
    print(f"{name:<12} linkability  risk = {r.value:.4f}  CI = "
          f"({r.ci[0]:.4f}, {r.ci[1]:.4f})")
    return {"model": name, "attack": "linkability",
            "risk": r.value, "ci_low": r.ci[0], "ci_high": r.ci[1]}

link_rows = [linkability(syn, name)
             for name, syn in [("CTGAN", syn_ctgan), ("CTAB-GAN+", syn_ctab)]]


CTGAN        linkability  risk = 0.0047  CI = (0.0000, 0.0131)
CTAB-GAN+    linkability  risk = 0.0081  CI = (0.0000, 0.0181)


## 8. Inference risk (per attribute)

For every column treated as the secret, the attacker uses all other columns as
auxiliary knowledge and tries to infer the secret from the synthetic data. This
yields a per-attribute risk profile, highlighting which variables leak the most
information. The target `def_pay` is included automatically as one of the
secrets.


In [9]:
def inference_all(syn, name):
    rows = []
    for secret in ALL_COLS:
        aux = [c for c in ALL_COLS if c != secret]
        ev = InferenceEvaluator(ori=ori, syn=syn, control=control,
                                n_attacks=N_ATTACKS,
                                aux_cols=aux, secret=secret)
        ev.evaluate()
        r = ev.risk()
        rows.append({"model": name, "attack": "inference", "secret": secret,
                     "risk": r.value, "ci_low": r.ci[0], "ci_high": r.ci[1]})
    return rows

inf_rows = []
for name, syn in [("CTGAN", syn_ctgan), ("CTAB-GAN+", syn_ctab)]:
    print(f"\nInference attacks for {name} (one per column)...")
    rows = inference_all(syn, name)
    inf_rows.extend(rows)
    # quick per-model summary: top-5 most vulnerable attributes
    top = (pd.DataFrame(rows).sort_values("risk", ascending=False)
           .head(5)[["secret", "risk"]])
    print(top.to_string(index=False))



Inference attacks for CTGAN (one per column)...
  secret     risk
   PAY_6 0.120208
MARRIAGE 0.057030
     SEX 0.027744
   PAY_5 0.005264
PAY_AMT5 0.002229

Inference attacks for CTAB-GAN+ (one per column)...
   secret     risk
    PAY_6 0.065357
    PAY_2 0.049006
EDUCATION 0.045607
    PAY_5 0.042162
    PAY_3 0.025766


## 9. Consolidate and save results

Two CSVs are written, following the naming convention of the SynthEval and
Synthcity outputs:

* `anonymeter_summary_results.csv` — singling out + linkability, one row per
  attack and model.
* `anonymeter_inference_results.csv` — per-attribute inference risk.


In [10]:
summary_df = pd.DataFrame(so_rows + link_rows)
inference_df = pd.DataFrame(inf_rows)

summary_df.to_csv(DATA_DIR / "anonymeter_summary_results.csv", index=False)
inference_df.to_csv(DATA_DIR / "anonymeter_inference_results.csv", index=False)
print("Saved: anonymeter_summary_results.csv")
print("Saved: anonymeter_inference_results.csv")

print("\n=== SINGLING OUT + LINKABILITY ===")
print(summary_df.pivot(index="model", columns="attack", values="risk").round(4))

print("\n=== INFERENCE (mean risk per model) ===")
print(inference_df.groupby("model")["risk"].mean().round(4))


Saved: anonymeter_summary_results.csv
Saved: anonymeter_inference_results.csv

=== SINGLING OUT + LINKABILITY ===
attack     linkability  singling_out_multivariate  singling_out_univariate
model                                                                     
CTAB-GAN+       0.0081                     0.0792                   0.0262
CTGAN           0.0047                     0.0487                   0.0000

=== INFERENCE (mean risk per model) ===
model
CTAB-GAN+    0.0105
CTGAN        0.0089
Name: risk, dtype: float64


---
**Done.** If every cell ran without error, the two result CSVs are written and
the in-memory objects (`summary_df`, `inference_df`) are available for plotting
or further analysis. Any non-zero risk should be interpreted against its
confidence interval: a risk whose interval includes zero is not statistically
distinguishable from no leakage.
